In [ ]:
pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 34.3 MB/s eta 0:00:00


**Encoding using svt-av1**

In [ ]:
import av

def encode_video_svt_av1(input_file, output_file, crf=None, gop_size=None, use_b_frames=False, preset=8):
    """Decode and encode the video with SVT-AV1 codec with speed optimizations."""
    input_container = av.open(input_file)
    output_container = av.open(output_file, mode='w')

    input_stream = next(s for s in input_container.streams if s.type == 'video')
    frame_rate = int(input_stream.average_rate)

    # SVT-AV1 codec configuration
    output_stream = output_container.add_stream('libsvtav1', rate=frame_rate)
    output_stream.width = input_stream.codec_context.width
    output_stream.height = input_stream.codec_context.height
    output_stream.pix_fmt = 'yuv420p'

    # SVT-AV1 specific settings - optimized for speed
    output_stream.options = {
        'preset': str(preset),  # Higher preset values = faster encoding (0-13)
        'tune': '0',            # 0=VQ, 1=PSNR, 2=SSIM
        'film-grain': '0',      # Disable film grain synthesis
        'enable-overlays': '0', # Disable overlays
        'threads': '0',         # Use all available threads
        'tile-columns': '2',    # Use tile parallelism
        'tile-rows': '2',       # Use tile parallelism
    }

    if crf is not None:
        output_stream.options['crf'] = str(crf)
        output_stream.options['qp'] = str(crf)  # SVT-AV1 sometimes uses qp instead of crf

    if use_b_frames:
        output_stream.options['pred-struct'] = '2'  # Hierarchical prediction structure for B-frames
        output_stream.options['bias-pct'] = '50'     # Controls B-frame frequency (0-100)

    # Skip frame forcing completely to speed up processing
    for frame in input_container.decode(video=0):
        for packet in output_stream.encode(frame):
            output_container.mux(packet)

    # Flush encoder
    for packet in output_stream.encode():
        output_container.mux(packet)

    output_container.close()
    input_container.close()

def main():
    input_file = '/content/output.mp4'
    crf_value = 30
    preset = 10  # Higher preset (8-12) for much faster encoding

    output_file = f'new_output_svt_av1_fast_crf_{crf_value}.mkv'
    print(f"Encoding {output_file} with preset {preset}...")
    encode_video_svt_av1(input_file, output_file, crf=crf_value, preset=preset)

if __name__ == '__main__':
    main()

Encoding new_output_svt_av1_fast_crf_30.mkv with preset 10...


**Enabling Golden Frames**

In [ ]:
# import av
# import os
# import time
# from pathlib import Path

# def encode_with_svt_av1(input_path, output_path, crf, enable_golden_frames=True):
#     """
#     Decode an input video and re-encode it with SVT-AV1 codec with golden frames option.

#     Args:
#         input_path: Path to the input video
#         output_path: Path to save the encoded video
#         crf: Constant Rate Factor value
#         enable_golden_frames: Whether to enable golden frames

#     Returns:
#         dict: Encoding statistics
#     """
#     start_time = time.time()

#     # Open input video
#     input_container = av.open(input_path)
#     input_stream = input_container.streams.video[0]

#     # Get video properties
#     width = input_stream.width
#     height = input_stream.height
#     fps = input_stream.average_rate

#     # Create output container and stream
#     output_container = av.open(output_path, mode='w')

#     # Set codec to SVT-AV1
#     codec_name = 'libsvtav1'

#     # Create output stream
#     output_stream = output_container.add_stream(codec_name, rate=fps)
#     output_stream.width = width
#     output_stream.height = height
#     output_stream.pix_fmt = 'yuv420p'

#     # Set SVT-AV1 specific options
#     output_stream.options = {
#         'crf': str(crf),
#         'preset': '8',  # Medium preset (ranges from 0-13, where 0 is highest quality/slowest)
#     }

#     # Enable golden frames
#     if enable_golden_frames:
#         # Set SVT-AV1 parameters for golden frames
#         # The exact parameter name might vary based on the ffmpeg/pyAV version
#         # Common parameters that control golden frames in SVT-AV1
#         output_stream.options.update({
#             'svtav1-params': 'enable-golden-frame=1',
#             'enable-overlays': '1',  # Enable overlays which work with golden frames
#             'hierarchical-levels': '4',  # Higher levels work better with golden frames
#             'lookahead': '120',  # Longer lookahead helps with golden frame placement
#         })

#     frames_encoded = 0

#     # Process frames
#     for frame in input_container.decode(video=0):
#         # Encode frame
#         packets = output_stream.encode(frame)
#         for packet in packets:
#             output_container.mux(packet)
#         frames_encoded += 1

#     # Flush encoder
#     for packet in output_stream.encode():
#         output_container.mux(packet)

#     # Close containers
#     input_container.close()
#     output_container.close()

#     # Calculate encoding time and stats
#     end_time = time.time()
#     encoding_time = end_time - start_time

#     # Get output file size
#     output_size_bytes = os.path.getsize(output_path)
#     output_size_mb = output_size_bytes / (1024 * 1024)

#     stats = {
#         'crf': crf,
#         'golden_frames': enable_golden_frames,
#         'frames_encoded': frames_encoded,
#         'encoding_time_seconds': encoding_time,
#         'output_size_mb': output_size_mb,
#         'output_path': output_path
#     }

#     return stats

# def main():
#     # Input video path
#     input_video = "/content/output.mp4"

#     # Create output directory
#     output_dir = Path("/content/encoded_videos")
#     output_dir.mkdir(exist_ok=True)

#     # CRF values to test
#     crf_values = [30]

#     results = []

#     # Encode with golden frames for each CRF value
#     for crf in crf_values:
#         output_path = output_dir / f"svt_av1_golden_crf{crf}.mp4"
#         print(f"Encoding with CRF {crf} and golden frames enabled...")

#         stats = encode_with_svt_av1(
#             input_path=input_video,
#             output_path=str(output_path),
#             crf=crf,
#             enable_golden_frames=True
#         )

#         results.append(stats)
#         print(f"Completed encoding with CRF {crf}")
#         print(f"Output size: {stats['output_size_mb']:.2f} MB")
#         print(f"Encoding time: {stats['encoding_time_seconds']:.2f} seconds")
#         print("-" * 50)

#     # Print comparison results
#     print("\nEncoding Results Summary:")
#     print("-" * 80)
#     print(f"{'CRF':<5} | {'Golden Frames':<15} | {'Size (MB)':<10} | {'Time (s)':<10}")
#     print("-" * 80)

#     for stat in results:
#         print(f"{stat['crf']:<5} | {'Enabled':<15} | {stat['output_size_mb']:<10.2f} | {stat['encoding_time_seconds']:<10.2f}")

#     print("-" * 80)

# if __name__ == "__main__":
#     main()

Encoding with CRF 30 and golden frames enabled...


ValueError: [Errno 22] Invalid argument: '/content/encoded_videos/svt_av1_golden_crf30.mp4'

In [ ]:
import av
import os
import time
from pathlib import Path

def encode_with_svt_av1(input_path, output_path, crf, enable_golden_frames=True):
    """
    Decode an input video and re-encode it with SVT-AV1 codec with golden frames option.

    Args:
        input_path: Path to the input video
        output_path: Path to save the encoded video
        crf: Constant Rate Factor value
        enable_golden_frames: Whether to enable golden frames

    Returns:
        dict: Encoding statistics
    """
    start_time = time.time()

    # Open input video
    input_container = av.open(input_path)
    input_stream = input_container.streams.video[0]

    # Get video properties
    width = input_stream.width
    height = input_stream.height
    fps = input_stream.average_rate

    # Create output container and stream
    output_container = av.open(output_path, mode='w')

    # Set codec to SVT-AV1
    codec_name = 'libsvtav1'

    # Create output stream
    output_stream = output_container.add_stream(codec_name, rate=fps)
    output_stream.width = width
    output_stream.height = height
    output_stream.pix_fmt = 'yuv420p'

    # Base SVT-AV1 options
    output_stream.options = {
        'crf': str(crf),
        'preset': '8',  # Medium preset (ranges from 0-13, where 0 is highest quality/slowest)
    }

    # Enable golden frames - using individual parameters instead of svtav1-params
    if enable_golden_frames:
        # Set golden frame parameters directly
        output_stream.options.update({
            'hierarchical-levels': '4',  # Higher levels work better with golden frames
            'lookahead': '120',         # Longer lookahead helps with golden frame placement
            'keyint': '250',            # Controls I-frame (keyframe) interval
            'sc-detection': '1',        # Scene change detection
            'enable-overlays': '1'      # Enable overlays for golden frames
        })

        # Some versions of SVT-AV1 use these parameter names:
        if hasattr(output_stream, 'codec_context'):
            output_stream.codec_context.options.update({
                'g': '250',             # GOP size alternative parameter
                'enable-tf': '1'        # Enable temporal filtering
            })

    frames_encoded = 0

    # Process frames
    for frame in input_container.decode(video=0):
        # Encode frame
        packets = output_stream.encode(frame)
        for packet in packets:
            output_container.mux(packet)
        frames_encoded += 1

    # Flush encoder
    for packet in output_stream.encode():
        output_container.mux(packet)

    # Close containers
    input_container.close()
    output_container.close()

    # Calculate encoding time and stats
    end_time = time.time()
    encoding_time = end_time - start_time

    # Get output file size
    output_size_bytes = os.path.getsize(output_path)
    output_size_mb = output_size_bytes / (1024 * 1024)

    stats = {
        'crf': crf,
        'golden_frames': enable_golden_frames,
        'frames_encoded': frames_encoded,
        'encoding_time_seconds': encoding_time,
        'output_size_mb': output_size_mb,
        'output_path': output_path
    }

    return stats

def main():
    # Input video path
    input_video = "/content/output.mp4"

    # Create output directory
    output_dir = Path("/content/encoded_videos")
    output_dir.mkdir(exist_ok=True)

    # CRF values to test
    crf_values = [30]

    results = []

    # Try different output container formats if MP4 doesn't work
    for crf in crf_values:
        try:
            # First try with MP4
            output_path = output_dir / f"svt_av1_golden_crf{crf}.mp4"
            print(f"Encoding with CRF {crf} and golden frames enabled to MP4...")

            stats = encode_with_svt_av1(
                input_path=input_video,
                output_path=str(output_path),
                crf=crf,
                enable_golden_frames=True
            )

            results.append(stats)

        except ValueError as e:
            print(f"MP4 encoding failed: {e}")
            print("Trying MKV container instead...")

            # Try with MKV if MP4 fails
            output_path = output_dir / f"svt_av1_golden_crf{crf}.mkv"
            stats = encode_with_svt_av1(
                input_path=input_video,
                output_path=str(output_path),
                crf=crf,
                enable_golden_frames=True
            )

            results.append(stats)

        print(f"Completed encoding with CRF {crf}")
        print(f"Output size: {stats['output_size_mb']:.2f} MB")
        print(f"Encoding time: {stats['encoding_time_seconds']:.2f} seconds")
        print("-" * 50)

    # Print comparison results
    print("\nEncoding Results Summary:")
    print("-" * 80)
    print(f"{'CRF':<5} | {'Golden Frames':<15} | {'Size (MB)':<10} | {'Time (s)':<10}")
    print("-" * 80)

    for stat in results:
        print(f"{stat['crf']:<5} | {'Enabled':<15} | {stat['output_size_mb']:<10.2f} | {stat['encoding_time_seconds']:<10.2f}")

    print("-" * 80)

if __name__ == "__main__":
    main()

Encoding with CRF 30 and golden frames enabled to MP4...
MP4 encoding failed: [Errno 22] Invalid argument: '/content/encoded_videos/svt_av1_golden_crf30.mp4'
Trying MKV container instead...
Completed encoding with CRF 30
Output size: 766.27 MB
Encoding time: 5176.01 seconds
--------------------------------------------------

Encoding Results Summary:
--------------------------------------------------------------------------------
CRF   | Golden Frames   | Size (MB)  | Time (s)  
--------------------------------------------------------------------------------
30    | Enabled         | 766.27     | 5176.01   
--------------------------------------------------------------------------------


**Checking whether the output video contains golden frames or not?**

In [ ]:
import av
import os
import subprocess
import json
import re
import tempfile
from pathlib import Path

def detect_golden_frames_enhanced(video_path):
    """
    Enhanced detector for golden frames in SVT-AV1 encoded videos.
    Uses more specific methods to detect golden frame parameters.

    Args:
        video_path: Path to the SVT-AV1 encoded video

    Returns:
        dict: Information about golden frames in the video
    """
    print(f"Analyzing video: {video_path}")
    results = {}
    evidence = []

    # Check 1: Extract encoder settings using FFprobe (more detailed)
    encoder_cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-print_format', 'json',
        '-show_format',
        '-show_streams',
        video_path
    ]

    try:
        encoder_result = subprocess.run(encoder_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        video_info = json.loads(encoder_result.stdout)

        # Extract encoder settings from metadata if available
        encoder_settings = {}
        if 'streams' in video_info and len(video_info['streams']) > 0:
            stream = video_info['streams'][0]
            if 'tags' in stream:
                tags = stream['tags']

                # Look for encoder settings in tags
                for key, value in tags.items():
                    if 'encoder' in key.lower() or 'golden' in key.lower() or 'svt' in key.lower():
                        encoder_settings[key] = value

                        # Direct evidence if "golden" is mentioned in encoder settings
                        if 'golden' in value.lower():
                            evidence.append(f"Found direct reference to golden frames in encoder settings: {key}={value}")

        results['encoder_settings'] = encoder_settings

    except Exception as e:
        print(f"Error extracting encoder settings: {str(e)}")
        results['encoder_settings'] = {}

    # Check 2: Use FFmpeg with frame-specific analysis
    # This extracts more detailed AV1 coding information
    frame_analysis_cmd = [
        'ffmpeg',
        '-i', video_path,
        '-vf', 'showinfo',
        '-f', 'null',
        '-'
    ]

    try:
        frame_result = subprocess.run(frame_analysis_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        frame_output = frame_result.stderr.decode('utf-8')

        # Look for specific AV1 frame type indicators
        # More specific patterns for AV1 golden frames
        golden_patterns = [
            r'alt[_\s]*ref[_\s]*frame',
            r'golden[_\s]*frame',
            r'refresh[_\s]*frame[_\s]*flag',
            r'reference[_\s]*frame[_\s]*update',
            r'frame[_\s]*type[_\s]*:[_\s]*golden',
            r'use[_\s]*alt[_\s]*ref[_\s]*frame',
            r'frame[_\s]*update[_\s]*type'
        ]

        # Count matches for each pattern
        pattern_matches = {}
        for pattern in golden_patterns:
            matches = re.findall(pattern, frame_output, re.IGNORECASE)
            pattern_matches[pattern] = len(matches)

            if len(matches) > 0:
                evidence.append(f"Found {len(matches)} instances of '{pattern}' pattern in frame data")

        results['pattern_matches'] = pattern_matches

    except Exception as e:
        print(f"Error in frame analysis: {str(e)}")
        results['pattern_matches'] = {}

    # Check 3: Analyze frame distribution for evidence of golden frame usage
    # This looks at frame types and their distribution
    frames_cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-select_streams', 'v:0',
        '-show_frames',
        '-show_entries', 'frame=pict_type,key_frame,pkt_pts_time,best_effort_timestamp_time',
        '-of', 'json',
        video_path
    ]

    try:
        frames_result = subprocess.run(frames_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        frames_data = json.loads(frames_result.stdout)

        # Analyze frame patterns
        frames = frames_data.get('frames', [])
        total_frames = len(frames)

        # Extract keyframe positions
        keyframe_positions = []
        for i, frame in enumerate(frames):
            if frame.get('key_frame') == 1:
                keyframe_positions.append(i)

        # Calculate keyframe intervals
        keyframe_intervals = []
        for i in range(1, len(keyframe_positions)):
            interval = keyframe_positions[i] - keyframe_positions[i-1]
            keyframe_intervals.append(interval)

        # Look for evidence of golden frame pattern in keyframe intervals
        # (SVT-AV1 with golden frames tends to have a specific pattern)
        has_consistent_pattern = False
        if len(keyframe_intervals) > 2:
            # Calculate variation in intervals
            avg_interval = sum(keyframe_intervals) / len(keyframe_intervals)
            variations = [abs(interval - avg_interval) for interval in keyframe_intervals]
            avg_variation = sum(variations) / len(variations)

            # Low variation suggests algorithmic keyframe placement (often used with golden frames)
            if avg_variation < 10:
                has_consistent_pattern = True
                evidence.append(f"Found consistent keyframe pattern (avg interval: {avg_interval:.1f}, variation: {avg_variation:.1f}) suggesting structured reference frames")

        results['frame_analysis'] = {
            'total_frames': total_frames,
            'keyframe_count': len(keyframe_positions),
            'keyframe_ratio': len(keyframe_positions) / total_frames if total_frames > 0 else 0,
            'avg_keyframe_interval': sum(keyframe_intervals) / len(keyframe_intervals) if keyframe_intervals else 0,
            'has_consistent_pattern': has_consistent_pattern
        }

    except Exception as e:
        print(f"Error in keyframe analysis: {str(e)}")
        results['frame_analysis'] = {}

    # Check 4: Run bitstream analysis (specific to AV1)
    bitstream_cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-select_streams', 'v:0',
        '-show_entries', 'stream=codec_name,profile,level',
        '-of', 'json',
        video_path
    ]

    try:
        bitstream_result = subprocess.run(bitstream_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        bitstream_info = json.loads(bitstream_result.stdout)

        # Check if it's AV1
        is_av1 = False
        av1_info = {}

        if "streams" in bitstream_info and len(bitstream_info["streams"]) > 0:
            stream_info = bitstream_info["streams"][0]
            codec_name = stream_info.get("codec_name", "")

            if codec_name.lower() == "av1":
                is_av1 = True
                av1_info = {
                    "profile": stream_info.get("profile", "unknown"),
                    "level": stream_info.get("level", "unknown")
                }
                evidence.append(f"Video is encoded with AV1 codec (profile: {av1_info['profile']}, level: {av1_info['level']})")

        results['is_av1'] = is_av1
        results['av1_info'] = av1_info

    except Exception as e:
        print(f"Error in bitstream analysis: {str(e)}")
        results['is_av1'] = False
        results['av1_info'] = {}

    # Check 5: Direct parameter extraction using ffprobe or PyAV
    try:
        container = av.open(video_path)
        video_stream = container.streams.video[0]

        # Check for any codec parameters that might indicate golden frames
        codec_params = {}
        for key in dir(video_stream.codec_context):
            if not key.startswith('_'):
                try:
                    value = getattr(video_stream.codec_context, key)
                    if not callable(value) and value is not None:
                        codec_params[key] = str(value)

                        # Look for indicators in parameter names/values
                        if ('golden' in str(key).lower() or 'alt_ref' in str(key).lower() or
                            'golden' in str(value).lower() or 'alt_ref' in str(value).lower()):
                            evidence.append(f"Found golden frame indicator in codec parameter: {key}={value}")
                except:
                    pass

        container.close()
        results['codec_params'] = codec_params

    except Exception as e:
        print(f"Error in direct parameter extraction: {str(e)}")
        results['codec_params'] = {}

    # Final assessment
    # Determine if golden frames are likely present based on all evidence
    has_golden_frames = False
    confidence = "Low"

    # Count positive evidence
    positive_evidence_count = len(evidence)

    # Check for pattern matches which are strong indicators
    pattern_match_count = sum(results.get('pattern_matches', {}).values())

    if positive_evidence_count >= 3 or pattern_match_count > 0:
        has_golden_frames = True
        confidence = "High" if pattern_match_count > 0 else "Medium"
    elif results.get('is_av1', False) and results.get('frame_analysis', {}).get('has_consistent_pattern', False):
        has_golden_frames = True
        confidence = "Medium"
    elif results.get('is_av1', False):
        # Look at keyframe ratio - very low ratio (0.005-0.02) often indicates golden frames
        keyframe_ratio = results.get('frame_analysis', {}).get('keyframe_ratio', 0)
        if 0.005 <= keyframe_ratio <= 0.02:
            has_golden_frames = True
            confidence = "Medium"
            evidence.append(f"Keyframe ratio ({keyframe_ratio:.4f}) is consistent with golden frame usage")

    # Compile final results
    final_results = {
        "file_path": video_path,
        "is_av1_codec": results.get('is_av1', False),
        "av1_codec_info": results.get('av1_info', {}),
        "has_golden_frames": has_golden_frames,
        "confidence": confidence,
        "evidence": evidence,
        "frame_analysis": results.get('frame_analysis', {})
    }

    return final_results

def main():
    # Input video path - this should be one of your SVT-AV1 encoded videos
    input_video = '/content/encoded_videos/svt_av1_no_golden_crf18.mp4'

    # Run analysis
    results = detect_golden_frames_enhanced(input_video)

    # Print detailed results
    print("\nEnhanced Golden Frame Analysis Results:")
    print("=" * 60)
    print(f"File: {results['file_path']}")
    print(f"AV1 Codec: {'Yes' if results['is_av1_codec'] else 'No'}")

    if results['is_av1_codec']:
        print(f"AV1 Profile: {results['av1_codec_info'].get('profile', 'unknown')}")
        print(f"AV1 Level: {results['av1_codec_info'].get('level', 'unknown')}")

    if 'frame_analysis' in results:
        frame_analysis = results['frame_analysis']
        print(f"Total Frames: {frame_analysis.get('total_frames', 'N/A')}")
        print(f"Keyframes Count: {frame_analysis.get('keyframe_count', 'N/A')}")
        print(f"Keyframe Ratio: {frame_analysis.get('keyframe_ratio', 'N/A'):.4f}")
        print(f"Avg Keyframe Interval: {frame_analysis.get('avg_keyframe_interval', 'N/A'):.2f}")
        print(f"Consistent Frame Pattern: {'Yes' if frame_analysis.get('has_consistent_pattern', False) else 'No'}")

    print(f"Golden Frames Detected: {'Yes' if results['has_golden_frames'] else 'No'}")
    print(f"Confidence: {results['confidence']}")

    print("\nEvidence:")
    for item in results['evidence']:
        print(f"- {item}")

    if results['has_golden_frames']:
        print("\n✅ This video appears to contain golden frames.")
    else:
        print("\n❌ This video does not appear to contain golden frames or the evidence is insufficient.")

    print("\nNote: Golden frames are an internal feature of the AV1 codec.")
    print("This detector uses multiple approaches to infer their presence.")

# Add batch analysis function to check multiple videos
def batch_analysis(directory_path, pattern="svt_av1_*.mp4"):
    """
    Analyze multiple videos in a directory to check for golden frames.

    Args:
        directory_path: Path to directory containing encoded videos
        pattern: Glob pattern to match filenames
    """
    from glob import glob

    # Find all matching video files
    video_files = glob(os.path.join(directory_path, pattern))

    if not video_files:
        print(f"No videos matching pattern '{pattern}' found in {directory_path}")
        return

    print(f"Found {len(video_files)} videos to analyze")

    # Results summary
    results_summary = []

    # Analyze each video
    for video_path in video_files:
        print(f"\nAnalyzing: {os.path.basename(video_path)}")

        # Run the enhanced detector
        result = detect_golden_frames_enhanced(video_path)

        # Add to summary
        results_summary.append({
            'filename': os.path.basename(video_path),
            'has_golden_frames': result['has_golden_frames'],
            'confidence': result['confidence'],
            'evidence_count': len(result['evidence'])
        })

        # Print short result
        status = "✅" if result['has_golden_frames'] else "❌"
        print(f"{status} {os.path.basename(video_path)}: Golden frames {'detected' if result['has_golden_frames'] else 'not detected'} (Confidence: {result['confidence']})")

    # Print summary table
    print("\n=== Golden Frame Detection Summary ===")
    print(f"{'Filename':<40} | {'Golden Frames':<15} | {'Confidence':<10} | {'Evidence Count':<15}")
    print("-" * 85)

    for result in results_summary:
        print(f"{result['filename']:<40} | {'Detected' if result['has_golden_frames'] else 'Not Detected':<15} | {result['confidence']:<10} | {result['evidence_count']:<15}")

if __name__ == "__main__":
    # For single video analysis
    main()

    # # For analyzing all encoded videos
    # print("\n\nBatch analysis of all encoded videos:")
    # batch_analysis("/content/encoded_videos/", "svt_av1_*.mp4")

Analyzing video: /content/encoded_videos/svt_av1_no_golden_crf18.mp4

Enhanced Golden Frame Analysis Results:
File: /content/encoded_videos/svt_av1_no_golden_crf18.mp4
AV1 Codec: Yes
AV1 Profile: Main
AV1 Level: 8
Total Frames: 1542
Keyframes Count: 21
Keyframe Ratio: 0.0136
Avg Keyframe Interval: 75.00
Consistent Frame Pattern: Yes
Golden Frames Detected: Yes
Confidence: Medium

Evidence:
- Found consistent keyframe pattern (avg interval: 75.0, variation: 6.0) suggesting structured reference frames
- Video is encoded with AV1 codec (profile: Main, level: 8)

✅ This video appears to contain golden frames.

Note: Golden frames are an internal feature of the AV1 codec.
This detector uses multiple approaches to infer their presence.
